### Install requiremnts

In [1]:
!pip install -r ../requirements.txt


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import sys
sys.path.append('../')
from checkpoints import DT, MT, LT
import os


from validate_birdset import load_model, get_test_loader, test
from checkpoints import DT, MT, LT
import numpy as np
import requests
import zipfile
import glob

DOWN_TASKS = ["HSN"]
DEVICE = "cuda" # change to "cpu" if no gpu is available.

/home/bellafkir/Documents/sa4birds/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Download main checkpoints

In [2]:
def download_ckpt(url, extract_dir):
   
    zip_filename = "file.zip"
    
    # -----------------------
    # Download ZIP file
    # -----------------------
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(zip_filename, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

    print("Download complete.")

    # -----------------------
    # Extract ZIP file
    # -----------------------
    os.makedirs(extract_dir, exist_ok=True)

    with zipfile.ZipFile(zip_filename, "r") as zip_ref:
        zip_ref.extractall(extract_dir)

    print(f"Unzipped to '{extract_dir}'")
    

In [3]:
def run_testing(ckpts, extracted_interval=7, target_length=701, device='cuda'):
    for down_task in DOWN_TASKS:

        print(ckpts)

        models = []
        configs = []

        # ---------------------------------------------------------
        # Load models and their configs from checkpoints
        # ---------------------------------------------------------
        for ckpt in ckpts:
            m, cfg = load_model(ckpt, device=device)
            models.append(m)
            # Override validation parameters for evaluation
            cfg.frontend.val_target_length = target_length
            cfg.event_decoder.val.extracted_interval = extracted_interval
            # Set dataset name to current downstream task
            cfg.train.dataset_name = down_task
            configs.append(cfg)

        results = dict()
        # Create test dataloader using the first config
        val_loader, class_names = get_test_loader(configs[0])

        # Map class names to label IDs using the config label map
        label2id = {k: v for k, v in configs[0].train.label_map.items()}

        # Only test on classes relevant for the current dataset
        relevant = [label2id[c] for c in class_names]

        # Store metrics for each model (for later averaging)
        metrics = {"auroc": [],
                   "cmap": [],
                   "top1_acc": []}

        # ---------------------------------------------------------
        # test each model on the test dataset
        # ---------------------------------------------------------
        for model, _ in zip(models, configs):
            # Run evaluation
            auroc, cmap, top1_acc = test(model, val_loader, relevant, device=device)

            # Store metrics
            metrics["auroc"].append(auroc)
            metrics["cmap"].append(cmap)
            metrics["top1_acc"].append(top1_acc)

        # ---------------------------------------------------------
        # Aggregate metrics across all checkpoints/models
        # ---------------------------------------------------------
        results[down_task] = {key: float(np.mean(values)) for key, values in metrics.items()}
        print(results)

## 1. Impact of incorporating secondary label annotations into training, comparing model evaluation metrics with and without annotation weighting on the HSN test set (Table 5)

In [5]:
# download/load checkpoints trained without secondary labels
if not os.path.exists('../ckpts/rest/HSN_secw0'):
    download_ckpt('https://next.hessenbox.de/index.php/s/7KLBQTNGK6o79Z5/download', extract_dir="../ckpts/rest/")

# get ckpts paths 
ckpts = glob.glob('../ckpts/rest/HSN_secw0/*')

run_testing(ckpts, device=DEVICE)
    

['../checkpoints/rest/HSN_secw0/HSN_eca_nfnet_l1_2026-03-09_122507', '../checkpoints/rest/HSN_secw0/HSN_eca_nfnet_l1_2026-03-09_124840', '../checkpoints/rest/HSN_secw0/HSN_eca_nfnet_l1_2026-03-09_123657']


2026-03-14 10:34:11,046 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:34:11,202 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 10:34:11,219 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 10:34:11,685 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:34:11,993 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 10:34:12,008 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 10:34:12,489 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:34:12,606 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9502559753896435, 'cmap': 0.5623522065687582, 'top1_acc': 0.7183063626289368}}


In [6]:
if not os.path.exists('../ckpts/DT/HSN'):
    download_ckpt('https://next.hessenbox.de/index.php/s/KR92DHDjYCSMREc/download', extract_dir="../ckpts/DT/")

# with secondary labels 
ckpts = glob.glob('../ckpts/DT/HSN/*')
run_testing(ckpts, device=DEVICE)

['../checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_113316', '../checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_112131', '../checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_114501']


2026-03-14 10:36:34,618 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:36:34,734 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 10:36:34,747 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 10:36:35,149 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:36:35,262 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 10:36:35,274 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 10:36:35,679 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:36:35,793 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9560262623085944, 'cmap': 0.5792627376801449, 'top1_acc': 0.7324406305948893}}


## 2. Performance metrics for various classifier heads tested on the HSN (Table 4)

In [7]:
# DSA 
ckpts = glob.glob('../ckpts/DT/HSN/*')
run_testing(ckpts, device=DEVICE)

['../checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_113316', '../checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_112131', '../checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_114501']


2026-03-14 10:38:59,479 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:38:59,639 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 10:38:59,652 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 10:39:00,017 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:39:00,135 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 10:39:00,149 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 10:39:00,518 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:39:00,635 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9560262623085944, 'cmap': 0.5792627376801449, 'top1_acc': 0.7324406305948893}}


In [9]:
# Linear

if not os.path.exists('../ckpts/rest/HSN_linearhead'):
    download_ckpt('https://next.hessenbox.de/index.php/s/HzFNW7SS4fA8Spe/download', extract_dir="../checkpoints/rest/")

ckpts = glob.glob('../ckpts/rest/HSN_linearhead/*')
run_testing(ckpts, device=DEVICE)

['../checkpoints/rest/HSN_linearhead/HSN_eca_nfnet_l1_2026-03-09_153017', '../checkpoints/rest/HSN_linearhead/HSN_eca_nfnet_l1_2026-03-09_151901', '../checkpoints/rest/HSN_linearhead/HSN_eca_nfnet_l1_2026-03-09_150746']


2026-03-14 10:46:27,199 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:46:27,386 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 10:46:27,400 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 10:46:27,746 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:46:27,870 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 10:46:27,878 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 10:46:28,238 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:46:28,357 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9120344598123685, 'cmap': 0.5234318648468855, 'top1_acc': 0.6881780425707499}}


In [10]:
run_testing(ckpts, extracted_interval=5, target_length=501, device=DEVICE)

['../checkpoints/rest/HSN_linearhead/HSN_eca_nfnet_l1_2026-03-09_153017', '../checkpoints/rest/HSN_linearhead/HSN_eca_nfnet_l1_2026-03-09_151901', '../checkpoints/rest/HSN_linearhead/HSN_eca_nfnet_l1_2026-03-09_150746']


2026-03-14 10:48:45,415 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:48:45,536 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 10:48:45,547 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 10:48:45,849 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:48:45,967 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 10:48:45,978 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 10:48:46,272 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:48:46,401 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9261919057612878, 'cmap': 0.5521745045732435, 'top1_acc': 0.6725559631983439}}


In [11]:
# Time Attention
if not os.path.exists('../ckpts/rest/HSN_timeattention'):
    download_ckpt('https://next.hessenbox.de/index.php/s/5KWi6EtFNm87Z7L/download', extract_dir="../ckpts/rest/")

ckpts = glob.glob('../ckpts/rest/HSN_timeattention/*')


run_testing(ckpts, device=DEVICE)

['../checkpoints/rest/HSN_timeattention/HSN_eca_nfnet_l1_2026-03-09_162312', '../checkpoints/rest/HSN_timeattention/HSN_eca_nfnet_l1_2026-03-09_161144', '../checkpoints/rest/HSN_timeattention/HSN_eca_nfnet_l1_2026-03-09_163440']


2026-03-14 10:50:28,570 | INFO | Task: HSN | Number of events: 12000
Testing: 100%|██████████| 94/94 [00:43<00:00,  2.14it/s]


{'HSN': {'auroc': 0.9457022326757826, 'cmap': 0.5847186219686932, 'top1_acc': 0.7046060164769491}}


In [4]:
# SSA
if not os.path.exists('../ckpts/rest/HSN_SSA'):
    download_ckpt('https://next.hessenbox.de/index.php/s/j6xrKx5ajQQQpZL/download', extract_dir="../ckpts/rest/")

ckpts = glob.glob('../ckpts/rest/HSN_SSA/*')


run_testing(ckpts, device=DEVICE)

['../checkpoints/rest/HSN_SSA/HSN_eca_nfnet_l1_2026-03-14_105101', '../checkpoints/rest/HSN_SSA/HSN_eca_nfnet_l1_2026-03-14_102758', '../checkpoints/rest/HSN_SSA/HSN_eca_nfnet_l1_2026-03-14_103929']


2026-03-14 12:25:06,051 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 12:25:06,210 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 12:25:06,224 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 12:25:06,582 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 12:25:06,692 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 12:25:06,702 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 12:25:07,058 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 12:25:07,175 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9551479085393016, 'cmap': 0.5842041371826746, 'top1_acc': 0.724009652932485}}


## 3. Effect of temporal context length during testing, showing how segment extension enhances model performance (Table 6)

In [12]:
# DFA using a 7-second input, where the centered 5 seconds are the target samples.
# target_length is the time resolution of the spectrograms
ckpts = glob.glob('../ckpts/DT/HSN/*')
run_testing(ckpts, extracted_interval=7, target_length=701, device=DEVICE)

['../checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_113316', '../checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_112131', '../checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_114501']


2026-03-14 10:52:44,518 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:52:44,645 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 10:52:44,659 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 10:52:45,024 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:52:45,143 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 10:52:45,156 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 10:52:45,553 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:52:45,669 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9560262623085944, 'cmap': 0.5792627376801449, 'top1_acc': 0.7324406305948893}}


In [13]:
# DFA using the original 5-second input of the test samples.
run_testing(ckpts, extracted_interval=5, target_length=501, device=DEVICE)

['../checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_113316', '../checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_112131', '../checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_114501']


2026-03-14 10:55:05,244 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:55:05,362 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 10:55:05,374 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 10:55:05,753 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:55:05,871 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 10:55:05,882 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 10:55:06,273 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:55:06,392 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9524781767262779, 'cmap': 0.5791612980916198, 'top1_acc': 0.7189263105392456}}


## 4. Ablation study on data augmentation methods, evaluating their impact on bird species recognition using HSN test data (Table 7)

In [14]:
# Baseline: No Augmentation

if not os.path.exists('../ckpts/rest/HSN_noaug'):
    download_ckpt('https://next.hessenbox.de/index.php/s/CqqJCewPWkYwoHA/download', extract_dir="../ckpts/rest/")

ckpts = glob.glob('../ckpts/rest/HSN_noaug/*')
run_testing(ckpts, device=DEVICE)

['../checkpoints/rest/HSN_noaug/HSN_eca_nfnet_l1_2025-10-27_084621', '../checkpoints/rest/HSN_noaug/HSN_eca_nfnet_l1_2025-10-27_083456', '../checkpoints/rest/HSN_noaug/HSN_eca_nfnet_l1_2025-10-27_085747']


2026-03-14 10:56:51,741 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:56:51,858 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 10:56:51,872 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 10:56:52,300 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:56:52,421 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 10:56:52,433 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 10:56:52,964 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:56:53,082 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.8588290967953124, 'cmap': 0.5249688989157181, 'top1_acc': 0.6417457063992819}}


In [15]:
# +Mixup (Signal)

if not os.path.exists('../ckpts/rest/HSN_sigmixup'):
    download_ckpt('https://next.hessenbox.de/index.php/s/HNXWy6Hx9NpbJXc/download', extract_dir="../ckpts/rest/")

ckpts = glob.glob('../ckpts/rest/HSN_sigmixup/*')
run_testing(ckpts, device=DEVICE)

['../checkpoints/rest/HSN_sigmixup/HSN_eca_nfnet_l1_2025-10-27_100822', '../checkpoints/rest/HSN_sigmixup/HSN_eca_nfnet_l1_2025-10-27_095651', '../checkpoints/rest/HSN_sigmixup/HSN_eca_nfnet_l1_2025-10-27_101953']


2026-03-14 10:59:14,738 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:59:14,873 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 10:59:14,887 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 10:59:15,315 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:59:15,430 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 10:59:15,443 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 10:59:15,870 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 10:59:15,990 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9059694394473373, 'cmap': 0.5447450570350201, 'top1_acc': 0.7146488030751547}}


In [16]:
# +Mixup (Signal) +Background-noise

if not os.path.exists('../ckpts/rest/HSN_sigmixup+bgnoise'):
    download_ckpt('https://next.hessenbox.de/index.php/s/KoMSXJKrF8YDPtD/download', extract_dir="../ckpts/rest/")

ckpts = glob.glob('../ckpts/rest/HSN_sigmixup+bgnoise/*')
run_testing(ckpts, device=DEVICE)

['../checkpoints/rest/HSN_sigmixup+bgnoise/HSN_eca_nfnet_l1_2025-10-27_103302', '../checkpoints/rest/HSN_sigmixup+bgnoise/HSN_eca_nfnet_l1_2025-10-27_105614', '../checkpoints/rest/HSN_sigmixup+bgnoise/HSN_eca_nfnet_l1_2025-10-27_104438']


2026-03-14 11:01:35,273 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 11:01:35,396 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 11:01:35,410 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 11:01:35,820 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 11:01:35,942 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 11:01:35,956 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 11:01:36,386 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 11:01:36,506 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9062293678286514, 'cmap': 0.5517955212866806, 'top1_acc': 0.7202281554539999}}


In [17]:
# +Mixup (Signal) +Background noise +colored noise +gain

if not os.path.exists('../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain'):
    download_ckpt('https://next.hessenbox.de/index.php/s/ELkWyHidSpSxN69/download', extract_dir="../ckpts/rest/")

ckpts = glob.glob('../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain/*')
run_testing(ckpts, device=DEVICE)

['../checkpoints/rest/HSN_sigmixup+bgnoise+colorednoise+gain/HSN_eca_nfnet_l1_2026-03-09_205253', '../checkpoints/rest/HSN_sigmixup+bgnoise+colorednoise+gain/HSN_eca_nfnet_l1_2026-03-09_203948', '../checkpoints/rest/HSN_sigmixup+bgnoise+colorednoise+gain/HSN_eca_nfnet_l1_2026-03-09_210951']


2026-03-14 11:03:55,439 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 11:03:55,565 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 11:03:55,579 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 11:03:56,000 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 11:03:56,119 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 11:03:56,135 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 11:03:56,563 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 11:03:56,675 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9115621064078656, 'cmap': 0.5518943480274384, 'top1_acc': 0.7261794209480286}}


In [18]:
# +Mixup (Signal) +Background-noise +colored-noise +gain +no-call 

if not os.path.exists('../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain+nocall'):
    download_ckpt('https://next.hessenbox.de/index.php/s/mH24adMWiSZ6HeE/download', extract_dir="../ckpts/rest/")

ckpts = glob.glob('../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain+nocall/*')
run_testing(ckpts, device=DEVICE)

['../checkpoints/rest/HSN_sigmixup+bgnoise+colorednoise+gain+nocall/HSN_eca_nfnet_l1_2026-03-10_113228', '../checkpoints/rest/HSN_sigmixup+bgnoise+colorednoise+gain+nocall/HSN_eca_nfnet_l1_2026-03-10_115551', '../checkpoints/rest/HSN_sigmixup+bgnoise+colorednoise+gain+nocall/HSN_eca_nfnet_l1_2026-03-10_114410']


2026-03-14 11:06:15,040 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 11:06:15,160 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 11:06:15,174 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 11:06:15,599 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 11:06:15,713 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 11:06:15,727 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 11:06:16,147 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 11:06:16,374 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9506547736325318, 'cmap': 0.5801166357128348, 'top1_acc': 0.7243196368217468}}


In [19]:
# +Mixup (Signal) +Background-noise +colored-noise +gain +no-call +Mixup (Spec) +SpecAug

if not os.path.exists('../ckpts/DT/HSN'):
    download_ckpt('https://next.hessenbox.de/index.php/s/KR92DHDjYCSMREc/download', extract_dir="../ckpts/DT/")

# with secondary labels 
ckpts = glob.glob('../ckpts/DT/HSN/*')
run_testing(ckpts, device=DEVICE)

['../checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_113316', '../checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_112131', '../checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_114501']


2026-03-14 11:08:35,768 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 11:08:35,886 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 11:08:35,892 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 11:08:36,260 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 11:08:36,377 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 11:08:36,391 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 11:08:36,752 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 11:08:36,871 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9560262623085944, 'cmap': 0.5792627376801449, 'top1_acc': 0.7324406305948893}}
